# Lab 22: Clustering Economies — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 68 min

---

**Learning Objectives:**
- Find three errors in an analyst's K-Means pipeline, one in each of its three steps, and say which printed numbers show each one
- Correct Step 1 (one line)
- Write `evaluate_k_range()`, a function that scores K-Means at every k by WCSS and silhouette, and use its table to choose the number of clusters
- Say what the clusters you choose separate, in economic terms, and compare them with the World Bank's income groups

**Dataset:** World Bank World Development Indicators: ten indicators for 209 economies, downloaded with `wbgapi` on 4 October 2026 and saved, so everyone works on the same numbers. The World Bank's regional and income-group aggregates (`World`, `Euro area`) and the economies missing more than four of the ten indicators were removed before saving. It is the download ECON 3916's Lab 22 uses.

**Format:** Part 1 runs an analyst's pipeline in three steps, with one error in each, and asks what each error is. Part 2 corrects Step 1. Write it yourself redoes Step 2's work as a function, and the cells after it use your function to choose k. Write your answers in the text cells.

**Deliverable:** your three diagnoses (Part 1), the blank in Part 2, the required `evaluate_k_range()` function, and your choice of k in two or three sentences.

**Time estimate:** ~68 minutes: setup 6, Part 1 20, Part 2 4, Write it yourself 20, choosing k 9, and the README and the GitHub submission 9. There is no optional part.

**AI in this lab: no AI code.** You may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. Every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Part 2:** one blank, which corrects Step 1 (one line).
> - **Write it yourself (required):** `evaluate_k_range()`, a function that fits K-Means at each k and returns a table of WCSS and silhouette, about eight lines in an empty cell. A check cell tells you when it is right.
>
> **You write, in text cells:** three answers in Part 1, and two or three sentences choosing k at the end of Write it yourself.
>
> **You run and read:** everything else. That is loading the saved data, a short example of `StandardScaler`, the analyst's three steps, and the table, chart and cluster profiles made with your function. Read each cell before you run it: the questions ask about what it prints. The check cell is a tool: run it and read what it prints; its code is not part of the lab. The README is text only: the prompt at the end is for documentation, not code.

## Setup (6 min)

The first line installs scikit-learn 1.8 or 1.9, and a matplotlib below 3.12, if Colab has
other versions; the `!` runs the line as a terminal command, not as Python. The expected numbers in
this lab were produced on scikit-learn 1.8 and 1.9. The cell prints your scikit-learn version. If it
stops with an `ImportError`, or shows another version, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries this lab uses
!pip install "scikit-learn>=1.8,<1.10" "matplotlib>=3.8,<3.12" -q

import hashlib
import io
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

print(f"scikit-learn {sklearn.__version__} (this lab needs 1.8 or 1.9)")
print("Setup complete.")

**Loading the saved data.** The World Bank revises its numbers and adds new ones every week, so
the data were saved once, as a compressed CSV file, and every copy of this lab reads that file.
- `urlopen(DATA_URL)` opens the file on the course's GitHub page, and `.read()` downloads it as raw
  bytes. `with ... as response:` closes the connection when the indented block ends.
- `try` runs the lines under it; if they raise an error (no internet, say), `except` runs instead
  and looks for a copy in a `data` folder beside the notebook. That copy exists only in a downloaded
  copy of the course repository, not in Colab. If neither works, the cell stops with a message
  saying what to do; `raise ... from None` shows that message without the error behind it.
- `hashlib.md5(data).hexdigest()` is a fingerprint of the file: 32 characters that change if a
  single byte of the file changes. The cell compares it with the fingerprint of the file this lab
  was written on, so a changed file stops here instead of giving numbers that differ from the
  expected outputs.
- `pd.read_csv(io.BytesIO(data), compression="gzip", ...)` reads the downloaded bytes as a
  compressed CSV file. `index_col="economy"` makes the three-letter codes (`USA`, `NGA`) the row
  labels, and `float_precision="round_trip"` reads every number back exactly as it was saved.

In [ ]:
# GIVEN — run as-is
# Load the saved World Bank data: one row per economy
DATA_URL = ("https://raw.githubusercontent.com/RyanPiao/econ5200-student-notebooks/main/"
            "5200/data/lab-ch22-wdi-2026-10.csv.gz")
LOCAL_COPY = Path("data/lab-ch22-wdi-2026-10.csv.gz")
DATA_MD5 = "454b3bb335718d430c9da98661e2f693"     # the fingerprint of the file this lab was written on

try:
    with urlopen(DATA_URL) as response:
        data = response.read()
except Exception:
    if not LOCAL_COPY.exists():
        raise RuntimeError("Could not download the saved World Bank data. Check that Colab says "
                           "Connected (top right), then run this cell again.") from None
    data = LOCAL_COPY.read_bytes()
if hashlib.md5(data).hexdigest() != DATA_MD5:
    raise RuntimeError("The data file is not the one this lab was written on, so your numbers would "
                       "not match the expected outputs. Run this cell again; if it stops here a "
                       "second time, tell your instructor.")

df = pd.read_csv(io.BytesIO(data), compression="gzip", index_col="economy",
                 float_precision="round_trip")
print(f"{len(df)} economies; columns: {', '.join(df.columns)}")

**Expected output:** `209 economies`, then the twelve columns: `Country`, the ten indicators, and
`income_level`, the World Bank's income group (LIC, LMC, UMC and HIC are low, lower-middle,
upper-middle and high income).

**The ten indicators.** GDP per person (in international dollars at purchasing power parity), life
expectancy (years), infant deaths per 1,000 births, primary school enrolment (a gross rate, which
can pass 100%), the Gini index of income inequality (0 to 100), CO2 emissions (tonnes per person),
internet users, trade (exports plus imports, as a share of GDP), unemployment, and the urban share
of the population, all in percent where not stated. Each is the economy's most recent reported
value, so two numbers in one row can come from different years.

**Reading the code.** K-Means cannot use a row with a blank in it, and 110 of the 2,090 values are
blank, 38 of them Gini indices. `.isna()` is True for a blank, and `.sum().sum()` counts the Trues
over every column. `.fillna(df[feature_names].median())` fills each blank with its column's median,
so 110 numbers below are not measurements, and 58 of the 209 economies have at least one filled
value. `.describe()` summarises each column; `.loc[["min", "50%", "max"]]` keeps its smallest value,
its median and its largest value, and `.T` turns the table so that each indicator is a row.

In [ ]:
# GIVEN — run as-is
# The ten indicators: count the blanks, fill each with its column's median, then each indicator's range
feature_names = ["gdp_per_capita_ppp", "life_expectancy", "infant_mortality", "primary_enrollment",
                 "gini_index", "co2_per_capita", "internet_users_pct", "trade_pct_gdp",
                 "unemployment_rate", "urban_population_pct"]
print(f"blanks filled with their column's median: {df[feature_names].isna().sum().sum()}")
df[feature_names] = df[feature_names].fillna(df[feature_names].median())
print(df[feature_names].describe().loc[["min", "50%", "max"]].T.round(1).to_string())

**Expected output:**
```
blanks filled with their column's median: 110
                         min      50%       max
gdp_per_capita_ppp    1154.9  21040.2  163354.0
life_expectancy         54.6     74.5      85.8
infant_mortality         1.2     12.5      71.9
primary_enrollment      20.9    100.2     184.0
gini_index              23.8     35.3      59.1
co2_per_capita           0.0      2.6      82.8
internet_users_pct       0.0     80.6     100.0
trade_pct_gdp            2.4     82.2     396.6
unemployment_rate        0.1      5.1      34.2
urban_population_pct    15.8     64.1     100.0
```

**A scikit-learn transformer, called on its own.** Labs 16 and 18 put `StandardScaler` inside a
pipeline, which called it for you. Called on its own, `StandardScaler().fit_transform(table)` does
two things: `fit` learns each column's mean and standard deviation, and `transform` subtracts the
mean from every value and divides by the standard deviation, so that every column has a mean of 0
and a standard deviation of 1. It returns a NumPy array. The next cell shows it on a three-row
table.

In [ ]:
# GIVEN — run as-is
# StandardScaler on its own, on a small table: each column to mean 0 and standard deviation 1
demo = pd.DataFrame({"a": [1.0, 2.0, 6.0], "b": [10.0, 40.0, 70.0]})
print(StandardScaler().fit_transform(demo).round(3))

**Expected output:**
```
[[-0.926 -1.225]
 [-0.463  0.   ]
 [ 1.389  1.225]]
```

---

## Part 1: DIAGNOSE — Find 3 Errors in This Clustering Pipeline (20 min)

An analyst wants to group the 209 economies by their development indicators with K-Means, and to
report how many groups there are. Her pipeline has three steps, one cell each, and each step has one
error. Every cell runs and prints numbers. Run the three cells, read what they print, and answer the
three questions after Step 3.

**Step 1, reading the code.** `.to_numpy()` gives the ten columns as a plain 209 × 10 array.
`KMeans(n_clusters=4, n_init=10, random_state=42)` sets up K-Means with four clusters, run from 10
random starts (it keeps the best), with starts that are the same on every run. `.fit(X)` fits it and
returns the fitted model; `.labels_` is each row's cluster number, 0 to 3 (a trailing `_` marks
something the fit learned). `np.bincount` counts the rows with each number, and
`df.groupby(labels)[feature_names].mean().T` averages each indicator within each cluster, one
column per cluster.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 1: the matrix of indicators, clustered into four groups
X = df[feature_names].to_numpy()
model = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
labels = model.labels_
print(f"economies per cluster: {np.bincount(labels)}")
print(df.groupby(labels)[feature_names].mean().T.round(1).to_string())

**Step 2, reading the code.** For each k from 2 to 8 the loop fits K-Means and appends a dictionary
of two scores to `rows`; `pd.DataFrame(rows)` makes one table row per dictionary.
- `model.inertia_` is the **WCSS**, the within-cluster sum of squares: each row's squared distance to
  its cluster's centroid, added up. Lower means tighter clusters.
- `silhouette_score(X, labels)` takes a matrix and a cluster number for each row. For each row it
  compares a, the average distance to the rest of its own cluster, with b, the average distance to
  the nearest other cluster, as (b − a) / max(a, b), and averages over the rows. Near 1, rows sit
  well inside their clusters; near 0, the clusters overlap.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 2: WCSS and silhouette for k = 2 to 8
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    rows.append({"k": k, "wcss": model.inertia_, "silhouette": silhouette_score(X, labels)})
scores = pd.DataFrame(rows)
print(scores.round(4).to_string(index=False))

**Step 3, reading the code.** `scores["wcss"].idxmin()` gives the row label of the smallest WCSS in
the table, and `scores.loc[row, "k"]` reads the k in that row.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 3: the number of clusters the analyst reports
best_k = scores.loc[scores["wcss"].idxmin(), "k"]
print(f"number of clusters reported: {best_k}")

### YOUR DIAGNOSIS

Each step has one error. For each step, say what the error is, which printed numbers show it, and
what it does to the analyst's conclusion. Quote numbers from the outputs above.

1. Step 1.
2. Step 2.
3. Step 3.

*Your answers here:*

---

## Part 2: FIX — Correct the Pipeline (4 min)

The pipeline is corrected one step at a time. Step 1 is corrected here. Write it yourself, next,
redoes Step 2's work as a function, and the cells after it use that function to do Step 3's job.

**Your task.** Fill in the blank with `X_fixed`, the matrix Step 1 should have clustered, made from
`df[feature_names]` (one line). The cell prints the matrix's shape, the largest of its ten column
means (as a distance from 0), and the smallest and largest of its ten column standard deviations.
`np.asarray(..., dtype=float)` makes sure it is measuring a NumPy array of numbers, and `np.abs`
takes absolute values.

In [ ]:
# YOUR TASK — Part 2: fill in the blank (____)
X_fixed = ____          # the matrix Step 1 should have clustered, from df[feature_names]
Z = np.asarray(X_fixed, dtype=float)
print(f"shape: {Z.shape}")
print(f"largest column mean, from 0: {np.abs(Z.mean(axis=0)).max():.4f}")
print(f"column standard deviations: {Z.std(axis=0).min():.4f} to {Z.std(axis=0).max():.4f}")

**Expected output:**
```
shape: (209, 10)
largest column mean, from 0: 0.0000
column standard deviations: 1.0000 to 1.0000
```
A `ValueError` that names a country (`could not convert string to float: 'Afghanistan'`) means
the matrix included the `Country` column: use `df[feature_names]`. Standard deviations of 0.9976
mean you divided by pandas' `.std()`, which divides by n − 1; `StandardScaler` divides by n. A
`TypeError` ending `not 'StandardScaler'` means `X_fixed` is the tool itself, not the matrix it
gives back.

---

## Write it yourself (required — 20 min)

This function is required. No AI: write it yourself.

Step 2 scored every k in one loop. Write that work as a function that any matrix and any list of k
can use. In the empty cell below, write

`evaluate_k_range(X, k_range, random_state=42)`

1. `X` is a matrix ready to cluster: one row per economy, one column per feature. Cluster it as it
   is, and do not change it.
2. `k_range` holds the numbers of clusters to try, such as `range(2, 9)`. Every k in it is at least 2.
3. For each k in `k_range`, in that order, fit `KMeans(n_clusters=k, n_init=10,
   random_state=random_state)` on `X`, and record k, the fit's WCSS and the silhouette score of that
   fit.
4. Return a pandas DataFrame with one row per k and the columns `k`, `wcss` and `silhouette`, in that
   order.

Inside the function, use its arguments, not variables from the lab's cells; the imports are fine.
`def evaluate_k_range(X, k_range, random_state=42):` starts the function, the indented lines under
it are its body, and `return` gives back the table. You need about eight lines.

The check runs your function on `X_fixed` with `range(2, 9)`; then three times on a small matrix of
its own, with `range(3, 6)` and `random_state=7`. Until you write the function, the check says "Not
written yet"; once you do, it fails with a message until your function is right. It takes a few
seconds.

In [ ]:
# WRITE IT YOURSELF — required: write evaluate_k_range(X, k_range, random_state=42)

In [ ]:
# CHECK — for evaluate_k_range above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "evaluate_k_range" not in globals():
    print("Not written yet: write evaluate_k_range in the empty cell above, then run this check again.")
else:
    import inspect

    def names_used(code):
        """Every global or attribute name a function's code refers to, including inner functions."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def strings_used(code):
        """Every string in a function's code, keyword names included, including inner functions."""
        found = set()
        for const in code.co_consts:
            if isinstance(const, str):
                found.add(const)
            elif isinstance(const, tuple):
                found |= {c for c in const if isinstance(c, str)}
            elif hasattr(const, "co_consts"):
                found |= strings_used(const)
        return found

    def near(a, b):
        """Equal to about six significant digits, element by element."""
        a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
        return a.shape == b.shape and bool(np.allclose(a, b, rtol=1e-6, atol=1e-9))

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(evaluate_k_range)
    params = list(signature.parameters)
    assert params[:3] == ["X", "k_range", "random_state"], (
        f"evaluate_k_range's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "evaluate_k_range(X, k_range, random_state=42).")
    assert signature.parameters["random_state"].default == 42, (
        "Give random_state a default of 42, as the task says.")

    used = names_used(evaluate_k_range.__code__)
    LAB_NAMES = {
        "labels": "labels, a variable from Step 1, instead of the cluster numbers of its own fit for "
                  "each k",
        "X_fixed": "X_fixed from Part 2 instead of its argument X",
        "df": "df, the lab's table, instead of its argument X",
        "feature_names": "feature_names, the lab's columns, instead of its argument X",
        "model": "model, the last fit from Step 2, instead of a fit of its own for each k",
        "scores": "scores, Step 2's table, instead of a table of its own",
        "rows": "rows, Step 2's list, instead of a list of its own",
        "best_k": "best_k from Step 3",
    }
    for name, what in LAB_NAMES.items():
        assert name not in used, f"evaluate_k_range uses {what}."
    if "KMeans" in used:
        keywords = strings_used(evaluate_k_range.__code__)
        assert "n_init" in keywords, (
            "evaluate_k_range's KMeans gets no n_init, so it runs from one start (n_init='auto' means one "
            "start) and can stop at a poor solution. Ask for n_init=10, as the task says.")
        assert "random_state" in keywords, (
            "evaluate_k_range's KMeans gets no random_state, so its starts change from run to run. Pass "
            "random_state=random_state.")
        assert "sample_size" not in keywords, (
            "evaluate_k_range scores the silhouette on a sample of the rows (sample_size). Score every row.")

    def run(where, X_in, k_range, **options):
        """Call evaluate_k_range on a copy of X_in; check the shape of what comes back; return it."""
        X_copy = np.array(X_in, dtype=float)
        try:
            got = evaluate_k_range(X_copy, k_range, **options)
        except ValueError as err:
            text = str(err)
            if "Number of labels is 1" in text:
                raise AssertionError(
                    f"{where}, evaluate_k_range stopped with ValueError: {text.rstrip('.')}. A silhouette needs at "
                    "least two clusters, so k = 1 cannot be scored. Loop over k_range itself; every k "
                    "in it is at least 2.") from None
            raise AssertionError(f"{where}, evaluate_k_range stopped with ValueError: {text}") from None
        except Exception as err:
            raise AssertionError(f"{where}, evaluate_k_range stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        assert np.array_equal(X_copy, np.asarray(X_in, dtype=float)), (
            f"{where}, evaluate_k_range changed the X it was given. Leave X as it is: no X -= ..., "
            "X /= ... or X[:] = ... inside the function.")
        assert got is not None, f"{where}, evaluate_k_range gives back nothing: use return, not print."
        if isinstance(got, list):
            raise AssertionError(f"{where}, evaluate_k_range gives back a list. Return a table: "
                                 "pd.DataFrame(rows), as Step 2 makes one.")
        if isinstance(got, dict):
            raise AssertionError(f"{where}, evaluate_k_range gives back a dictionary with keys "
                                 f"{', '.join(repr(key) for key in got)}. Return a table: pass the "
                                 "dictionary to pd.DataFrame.")
        if np.ndim(got) == 0:
            raise AssertionError(f"{where}, evaluate_k_range gives back one number, {got}. Return the "
                                 "whole table, one row per k.")
        assert isinstance(got, pd.DataFrame), (
            f"{where}, evaluate_k_range gives back a {type(got).__name__}. Return a pandas DataFrame.")
        cols = list(got.columns)
        if "k" not in cols and got.index.name == "k":
            raise AssertionError(f"{where}, k is the table's index, not a column. Keep it as a column "
                                 "(no set_index), so the columns are k, wcss and silhouette.")
        if sorted(cols) == ["k", "silhouette", "wcss"] and cols != ["k", "wcss", "silhouette"]:
            raise AssertionError(f"{where}, the columns come in the order {', '.join(cols)}. Put them in "
                                 "the order k, wcss, silhouette.")
        assert cols == ["k", "wcss", "silhouette"], (
            f"{where}, the table's columns are {cols}. Name them k, wcss and silhouette, in that order.")
        ks = list(k_range)
        got_k = [int(v) for v in got["k"]]
        if sorted(got_k) == sorted(ks) and got_k != ks:
            raise AssertionError(f"{where}, the rows come in the order k = {got_k}. Keep the order of "
                                 f"k_range, {ks}: do not sort the table.")
        if got_k == ks[:-1]:
            raise AssertionError(f"{where}, the table stops one k short: k = {got_k}, without "
                                 f"k = {ks[-1]}. Loop over k_range itself.")
        assert got_k == ks, (f"{where}, the table's k column is {got_k}; it should be {ks}, the k_range "
                             "the check passed. Loop over the k_range argument, not a range of your own.")
        if got["silhouette"].dtype == object:
            raise AssertionError(f"{where}, each silhouette is a whole array, one value per row. "
                                 "silhouette_score gives the average over the rows: use it.")
        return got

    # 1. The same arguments three times must give the same table
    gen = np.random.default_rng(2026)
    T = gen.uniform(0, 1, (150, 3)) * np.array([1.0, 10.0, 100.0])   # 150 rows, three columns
    where = "On the check's own matrix with range(3, 6) and random_state=7"
    tables = [run(where, T, range(3, 6), random_state=7) for _ in range(3)]
    assert all(near(t[["wcss", "silhouette"]], tables[0][["wcss", "silhouette"]]) for t in tables), (
        "Three calls with the same random_state gave different tables, so KMeans is not getting it. Pass "
        "random_state=random_state to KMeans.")

    # 2. X_fixed from Part 2, k = 2 to 8
    try:
        X_ready = np.asarray(X_fixed, dtype=float)
        ready = (X_ready.shape == (209, 10) and near(X_ready.mean(axis=0), np.zeros(10))
                 and near(X_ready.std(axis=0), np.ones(10)))
    except Exception:
        ready = False
    assert ready, ("X_fixed from Part 2 is not yet what Part 2 asks for, and this check uses it. Make "
                   "Part 2's output match its expected output, then run Part 2 and this check again.")
    MAXIMA = [4.088413015, 1.716797159, 3.28642205, 5.50524354, 3.605695418, 10.27661605, 1.11856586,
              5.460158058, 5.161465236, 1.660697278]          # X_fixed's largest value in each column
    assert near(X_ready.max(axis=0), MAXIMA), (
        "X_fixed has mean 0 and SD 1 but isn't df[feature_names] scaled as it is (a log or other change?). "
        "Make it from df[feature_names] alone, then run Part 2 and this check again.")
    WCSS = [1488.286346, 1289.773146, 1148.271198, 1041.158487, 979.5348794, 876.2389722, 812.3175011]
    SIL = [0.2729438194, 0.1846050075, 0.2012393381, 0.2040081132, 0.1860117361, 0.1870715094,
           0.1860670389]
    ONE_START = [1488.286346, 1293.227388, 1152.510138, 1078.771427, 1014.583446, 920.8306009,
                 852.2654184]
    where = "On X_fixed with range(2, 9)"
    got = run(where, X_ready, range(2, 9))
    wcss, sil = got["wcss"].to_numpy(dtype=float), got["silhouette"].to_numpy(dtype=float)
    assert not (near(wcss, SIL) and near(sil, WCSS)), (
        f"{where}, the wcss column holds the silhouettes and the silhouette column the WCSS: swap them.")
    assert not near(wcss, -np.array(WCSS)), (
        f"{where}, every WCSS is negative: .score() gives minus the WCSS. Use the fit's .inertia_.")
    assert not near(wcss, np.array(WCSS) / 209), (
        f"{where}, the WCSS is divided by the number of rows. .inertia_ is already the sum: use it as it is.")
    assert np.ptp(sil) > 1e-9, (
        f"{where}, the silhouette is {sil[0]:.4f} at every k, while the expected silhouettes differ from "
        "k to k. Check which cluster numbers your silhouette_score call receives.")
    assert not near(sil[1:], SIL[:-1]), (
        f"{where}, each k's silhouette is the one for the k before it. Score the fit you just made.")
    assert not near(wcss, ONE_START), (
        f"{where}, the WCSS at k = 3 to 8 is higher than ten starts reach (at k = 5, {wcss[3]:.2f} against "
        f"{WCSS[3]:.2f}): K-Means ran from one start. Ask for n_init=10.")
    got_k = [int(k) for k in got["k"]]
    wrong = [k for k, w, w_true in zip(got_k, wcss, WCSS) if not near(w, w_true)]
    assert not wrong, (
        f"{where}, the WCSS differs at k = {wrong}. Fit KMeans(n_clusters=k, n_init=10, "
        "random_state=random_state) on X itself, and record its .inertia_.")
    wrong = [k for k, s, s_true in zip(got_k, sil, SIL) if not near(s, s_true)]
    assert not wrong, (
        f"{where}, the WCSS is right but the silhouette differs at k = {wrong}. Score silhouette_score on "
        "X and the cluster numbers of the fit for that k, every row.")

    # 3. The check's own matrix: other k and another random_state
    where = "On the check's own matrix with range(3, 6) and random_state=7"
    wcss, sil = (tables[0][c].to_numpy(dtype=float) for c in ["wcss", "silhouette"])
    assert not near(wcss, [247.5864815, 197.7650552, 165.4303731]), (
        f"{where}, the numbers match this matrix with its columns rescaled, not the matrix as it was "
        "passed. The task gives evaluate_k_range a matrix ready to cluster: cluster X as it is.")
    assert not (near(wcss, [14947.43411, 9111.842833, 6167.047271])
                and near(sil, [0.5681710197, 0.5284993369, 0.5020874153])), (
        f"{where}, the table is the one for random_state=42: the function ignores its random_state "
        "argument. Pass random_state=random_state to KMeans.")
    assert near(wcss, [14947.43411, 9114.230279, 6164.948406]) and near(
        sil, [0.5681710197, 0.5270388453, 0.5052346352]), (
        f"{where}, the table differs from the one KMeans(n_clusters=k, n_init=10, random_state=7) gives. "
        "It passed on X_fixed, so look for a number that does not come from the arguments.")

    print("Passed: evaluate_k_range matches on X_fixed for k = 2 to 8, and on the check's own matrix.")

### Use it: choosing k (9 min)

The next cell runs your function on `X_fixed` for k = 2 to 8, prints the table, and draws each score
against k: the WCSS (the elbow plot) and the silhouette. `plt.subplots(1, 2, ...)` makes two charts
side by side, named `left` and `right`, and `.set(...)` labels a chart's axes and title in one call.

In [ ]:
# GIVEN — run as-is, once evaluate_k_range passes its check
# Your function on the corrected matrix, k = 2 to 8, and its two scores against k
scores_fixed = evaluate_k_range(X_fixed, range(2, 9))
print(scores_fixed.round(4).to_string(index=False))

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
left.plot(scores_fixed["k"], scores_fixed["wcss"], marker="o")
left.set(xlabel="k", ylabel="WCSS", title="WCSS (elbow plot)")
right.plot(scores_fixed["k"], scores_fixed["silhouette"], marker="o")
right.set(xlabel="k", ylabel="silhouette", title="Silhouette")
plt.tight_layout()
plt.show()

**Expected output:**
```
 k      wcss  silhouette
 2 1488.2863      0.2729
 3 1289.7731      0.1846
 4 1148.2712      0.2012
 5 1041.1585      0.2040
 6  979.5349      0.1860
 7  876.2390      0.1871
 8  812.3175      0.1861
```
and two charts of the same numbers. A WCSS of 1293.2274 at k = 3 means your function ran K-Means
from one start: rerun its check.

**What the clusters separate.** The next cell fits K-Means on `X_fixed` at k = 2 and at k = 4, the
number in the lecture's poll and the number of World Bank income groups. For each it prints the
clusters' sizes, the median of five indicators in each cluster, and how many economies of each World
Bank income group fall in each cluster. `.fit_predict(X_fixed)` fits and returns the cluster numbers
in one call, as `.fit(...).labels_` does. `pd.crosstab(a, b)` counts each pair of values;
`rownames=["cluster"]` labels its rows, and `[["LIC", "LMC", "UMC", "HIC"]]` puts the income groups
in order.

In [ ]:
# GIVEN — run as-is
# The clusters at k = 2 and k = 4: medians of five indicators, and the World Bank income groups
PROFILE = ["gdp_per_capita_ppp", "life_expectancy", "infant_mortality", "internet_users_pct",
           "unemployment_rate"]
for k in [2, 4]:
    groups = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_fixed)
    print(f"k = {k}: economies per cluster {np.bincount(groups)}")
    print(df.groupby(groups)[PROFILE].median().round(1).to_string())
    print(pd.crosstab(groups, df["income_level"], rownames=["cluster"])[["LIC", "LMC", "UMC", "HIC"]])
    print()

**Expected output:**
```
k = 2: economies per cluster [ 86 123]
   gdp_per_capita_ppp  life_expectancy  infant_mortality  internet_users_pct  unemployment_rate
0              6281.8             67.4              31.7                41.5                5.1
1             42295.2             78.0               7.5                89.5                5.1
income_level  LIC  LMC  UMC  HIC
cluster                         
0              25   39   20    2
1               0    8   39   76

k = 4: economies per cluster [90 56 13 50]
   gdp_per_capita_ppp  life_expectancy  infant_mortality  internet_users_pct  unemployment_rate
0             21040.2             75.2              11.7                82.0                5.3
1             68871.3             82.2               3.4                94.4                4.8
2             10251.3             66.2              33.8                63.3               19.9
3              4308.8             66.3              34.9                31.4                3.7
income_level  LIC  LMC  UMC  HIC
cluster                         
0               1   16   49   24
1               0    0    2   54
2               2    7    4    0
3              22   24    4    0
```

### YOUR JUDGMENT

In two or three sentences: which k would you report, and why? Weigh the silhouette against the WCSS
curve, say what the clusters at your k separate in economic terms, and use your own numbers.

*Your answer here:*

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR ...]` with your own words or the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Found three errors in an analyst's K-Means pipeline for 209
  economies and ten World Bank development indicators:
  [YOUR WORDS, one short line per error]
* Corrected the matrix she clustered, and wrote my own function
  that fits K-Means at every k and returns its WCSS and silhouette
* For k = 2 to 8, the silhouette ran from [YOUR VALUE] to
  [YOUR VALUE], highest at k = [YOUR VALUE]
* Chose k = [YOUR VALUE], because [YOUR REASON, one sentence]; the
  clusters separate [YOUR WORDS]

**Please write a README.md entry including:**
1. Project Title: Clustering Economies: Choosing k
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person, with no buzzwords. Use only
the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab22-clustering`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab22-clustering`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab22-clustering`, branch `main`,
   commit message `Lab 22 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.